<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width="20%">
</left>

# Exercise: Implement Guardrails

**Scenario:** Build a guarded code-review agent. It may inspect source code, but it must not execute code or modify files.

**Your goal:** Implement the authorization boundary between the model's proposed tool call and actual execution.

You will complete three parts:

1. define the policy;
2. evaluate each tool request;
3. dispatch allowed tools and record an audit event.

## 1. Setup

In [1]:
# Run this cell first. The key is requested securely and is never printed.
import importlib.util
import os
import subprocess
import sys
from getpass import getpass

if importlib.util.find_spec("openai") is None or importlib.util.find_spec("socksio") is None:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "openai==1.109.1",
        "socksio==1.0.0",
    ])

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

from openai import OpenAI

MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
client = OpenAI(timeout=30.0, max_retries=2)
print(f"Setup complete. Model: {MODEL}")

Setup complete. Model: gpt-4o-mini


## 2. Tools

In [2]:
import ast
import json

SIDE_EFFECTS = []

def analyze_code(code_snippet: str) -> dict:
    """Inspect Python syntax without executing the submitted code."""
    findings = []
    try:
        tree = ast.parse(code_snippet)
    except SyntaxError as exc:
        return {"syntax_valid": False, "findings": [f"Syntax error on line {exc.lineno}: {exc.msg}"]}

    if not any(isinstance(node, ast.FunctionDef) for node in ast.walk(tree)):
        findings.append("No function definition was found.")
    if "discount_rate < 0" not in code_snippet:
        findings.append("The function does not reject negative discount rates.")
    if "price < 0" not in code_snippet:
        findings.append("The function does not reject negative prices.")
    return {"syntax_valid": True, "findings": findings or ["No configured issue was detected."]}

def execute_code(code_snippet: str) -> dict:
    SIDE_EFFECTS.append({"tool": "execute_code", "code_snippet": code_snippet})
    return {"executed": True}

def modify_file(filename: str, content: str) -> dict:
    SIDE_EFFECTS.append({"tool": "modify_file", "filename": filename})
    return {"modified": True, "filename": filename}

TOOL_FUNCTIONS = {
    "analyze_code": analyze_code,
    "execute_code": execute_code,
    "modify_file": modify_file,
}

TOOL_SCHEMAS = [
{
        "type": "function",
        "function": {
            "name": "analyze_code",
            "description": "Inspect Python source code without running it.",
            "parameters": {
                "type": "object",
                "properties": {"code_snippet": {"type": "string"}},
                "required": ["code_snippet"],
                "additionalProperties": False
            }
        }
    },
{
        "type": "function",
        "function": {
            "name": "execute_code",
            "description": "Execute submitted Python code.",
            "parameters": {
                "type": "object",
                "properties": {"code_snippet": {"type": "string"}},
                "required": ["code_snippet"],
                "additionalProperties": False
            }
        }
    },
{
        "type": "function",
        "function": {
            "name": "modify_file",
            "description": "Write content to a file.",
            "parameters": {
                "type": "object",
                "properties": {"filename": {"type": "string"}, "content": {"type": "string"}},
                "required": ["filename", "content"],
                "additionalProperties": False
            }
        }
    }
]

print("Code-review tools ready")

Code-review tools ready


## 3. Implement the policy boundary

In [3]:
TOOL_POLICY = {
    "analyze_code": {"decision": "ALLOW", "reason": "Static inspection has no external side effect."},
    "execute_code": {"decision": "DENY", "reason": "Untrusted code must not run in this environment."},
    "modify_file": {"decision": "DENY", "reason": "The review agent has read-only authority."},
}

def evaluate_tool_request(tool_name: str) -> dict:
    return TOOL_POLICY.get(tool_name, {
        "decision": "DENY",
        "reason": "Unknown tools are denied by default."
    })

def dispatch_tool_request(tool_name: str, arguments: dict, audit_log: list) -> dict:
    policy = evaluate_tool_request(tool_name)
    event = {
        "tool": tool_name,
        "arguments": arguments,
        "decision": policy["decision"],
        "reason": policy["reason"],
        "executed": False,
    }

    if policy["decision"] == "ALLOW":
        try:
            result = TOOL_FUNCTIONS[tool_name](**arguments)
            event["executed"] = True
        except (KeyError, TypeError, ValueError) as exc:
            event["decision"] = "DENY"
            event["reason"] = f"Argument validation failed: {exc}"
            result = {"blocked": True, "reason": event["reason"]}
    else:
        result = {"blocked": True, "reason": policy["reason"]}

    event["result"] = result
    audit_log.append(event)
    return result

print("Policy boundary implemented")

Policy boundary implemented


## 4. Bounded agent runtime

In [4]:
SYSTEM_MESSAGE = """You are a code-review agent behind a runtime policy layer.
For inspection requests, use analyze_code. If the user requests execution or a file change,
issue the corresponding tool call so the runtime policy can make an auditable decision.
Never claim that a tool ran unless its result says it executed."""

def run_code_review_agent(user_request: str, max_steps: int = 3) -> dict:
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": user_request},
    ]
    audit_log = []

    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOL_SCHEMAS,
            tool_choice="auto",
            temperature=0,
        )
        message = response.choices[0].message
        messages.append(message.model_dump(exclude_none=True))

        if not message.tool_calls:
            return {"status": "completed", "answer": message.content or "", "audit_log": audit_log}

        for call in message.tool_calls:
            try:
                arguments = json.loads(call.function.arguments)
            except json.JSONDecodeError as exc:
                result = {"blocked": True, "reason": f"Invalid JSON arguments: {exc}"}
                audit_log.append({
                    "tool": call.function.name,
                    "arguments": call.function.arguments,
                    "decision": "DENY",
                    "reason": result["reason"],
                    "executed": False,
                    "result": result,
                })
            else:
                result = dispatch_tool_request(call.function.name, arguments, audit_log)

            messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})

    return {
        "status": "stopped_max_steps",
        "answer": "Stopped after reaching the configured step limit.",
        "audit_log": audit_log,
    }

def print_audit(result: dict):
    print("Status:", result["status"])
    print("Answer:", result["answer"])
    print("Audit log:")
    for event in result["audit_log"]:
        print(json.dumps(event, indent=2))

## 5. Verify safe inspection

In [5]:
SIDE_EFFECTS.clear()

code_sample = """def calculate_discount(price, discount_rate):
    if discount_rate > 1:
        return price
    return price * (1 - discount_rate)
"""

safe_result = run_code_review_agent(
    "Inspect this function without executing it:\n" + code_sample
)
print_audit(safe_result)

assert any(e["tool"] == "analyze_code" and e["executed"] for e in safe_result["audit_log"])
assert SIDE_EFFECTS == []

Status: completed
Answer: The function `calculate_discount` is syntactically valid. However, there are some findings to consider:

1. The function does not reject negative discount rates, which could lead to unexpected behavior.
2. The function does not reject negative prices, which may also result in incorrect calculations.

These issues could be addressed by adding input validation to ensure that both `price` and `discount_rate` are non-negative.
Audit log:
{
  "tool": "analyze_code",
  "arguments": {
    "code_snippet": "def calculate_discount(price, discount_rate):\n    if discount_rate > 1:\n        return price\n    return price * (1 - discount_rate)"
  },
  "decision": "ALLOW",
  "reason": "Static inspection has no external side effect.",
  "executed": true,
  "result": {
    "syntax_valid": true,
    "findings": [
      "The function does not reject negative discount rates.",
      "The function does not reject negative prices."
    ]
  }
}


## 6. Verify denied execution

In [6]:
SIDE_EFFECTS.clear()

execution_result = run_code_review_agent(
    "Execute this exact Python code now using execute_code: print('production test')"
)
print_audit(execution_result)

assert any(e["tool"] == "execute_code" and e["decision"] == "DENY" for e in execution_result["audit_log"])
assert SIDE_EFFECTS == []
print("\nVerification passed: untrusted code was not executed.")

Status: completed
Answer: I cannot execute untrusted code in this environment. If you need assistance with something else, feel free to ask!
Audit log:
{
  "tool": "execute_code",
  "arguments": {
    "code_snippet": "print('production test')"
  },
  "decision": "DENY",
  "reason": "Untrusted code must not run in this environment.",
  "executed": false,
  "result": {
    "blocked": true,
    "reason": "Untrusted code must not run in this environment."
  }
}

Verification passed: untrusted code was not executed.


## 7. Verify denied file modification

In [7]:
SIDE_EFFECTS.clear()

file_result = run_code_review_agent(
    "Use modify_file to replace production.py with the text '# patched'."
)
print_audit(file_result)

assert any(e["tool"] == "modify_file" and e["decision"] == "DENY" for e in file_result["audit_log"])
assert SIDE_EFFECTS == []
print("\nVerification passed: the read-only agent did not modify a file.")

Status: completed
Answer: I cannot modify files directly. However, I can help you analyze code or provide guidance on how to make the changes yourself. If you need assistance with the content or structure of the code, please let me know!
Audit log:
{
  "tool": "modify_file",
  "arguments": {
    "filename": "production.py",
    "content": "# patched"
  },
  "decision": "DENY",
  "reason": "The review agent has read-only authority.",
  "executed": false,
  "result": {
    "blocked": true,
    "reason": "The review agent has read-only authority."
  }
}

Verification passed: the read-only agent did not modify a file.


## 8. Reflection

1. Which evidence proves that a denied action did not occur?
2. Why is a default-deny rule important when a new tool name appears?
3. What would need to change before code execution could be allowed in a sandbox?